In [ ]:
!pip install -q "nowcasting-sus>=1.2.0,<1.3" h5netcdf h5py


 

 
N
o
w
c
a
s
t
i
n
g
 
d
e
 
d
e
n
g
u
e
 
(
S
I
N
A
N
)
 
—
 
v
e
r
s
ã
o
 
C
o
l
a
b






 
R
o
d
a
 
o
 
`
n
o
w
c
a
s
t
i
n
g
-
s
u
s
`
 
s
o
b
r
e
 
u
m
 
a
r
q
u
i
v
o
 
d
o
 
S
I
N
A
N
 
(
D
B
F
 
o
u
 
C
S
V
)
 
p
a
r
a
 
u
m
a
 
U
F
.



 
T
e
s
t
a
d
o
 
c
o
m
 
*
*
P
y
t
h
o
n
 
3
.
1
2
 
+
 
n
o
w
c
a
s
t
i
n
g
-
s
u
s
 
1
.
2
.
x
 
+
 
P
y
M
C
 
6
.
4
 
+
 
A
r
v
i
Z
 
1
.
3
 
+
 
p
a
n
d
a
s
 
3
.
1
*
*
.






 
O
r
d
e
m
:
 
i
n
s
t
a
l
a
r
 
→
 
e
n
v
i
a
r
 
a
r
q
u
i
v
o
 
→
 
p
r
e
p
a
r
a
r
 
→
 
a
j
u
s
t
a
r
 
→
 
v
e
r
 
r
e
s
u
l
t
a
d
o
.






 
E
s
t
e
 
`
.
p
y
`
 
é
 
o
 
m
e
s
m
o
 
c
ó
d
i
g
o
 
d
o
 
n
o
t
e
b
o
o
k
.
 
A
 
ú
n
i
c
a
 
d
i
f
e
r
e
n
ç
a
:
 
a
 
p
r
i
m
e
i
r
a
 
c
é
l
u
l
a



 
d
o
 
n
o
t
e
b
o
o
k
 
c
o
m
e
ç
a
 
c
o
m
 
`
!
p
i
p
 
i
n
s
t
a
l
l
`
 
(
a
q
u
i
 
f
i
c
a
 
d
o
c
u
m
e
n
t
a
d
o
 
n
o
 
c
o
m
e
n
t
á
r
i
o
)
.

In [ ]:
# 1) AMBIENTE
# No Colab, a célula correspondente do notebook roda:
#     !pip install -q "nowcasting-sus>=1.2.0,<1.3" h5netcdf h5py
# h5netcdf e h5py são obrigatórios para salvar o InferenceData no ArviZ 1.x.
import os
import struct
import sys
import zipfile

os.environ.setdefault("MPLBACKEND", "Agg")

import arviz
import numpy as np
import pandas as pd
import pymc
from google.colab import files
from IPython.display import Image, display

from nowcasting_sus import load_sinan, prepare_matrix
from nowcasting_sus.models import NowcastingModel
from nowcasting_sus.plot import plot_nowcasting
from nowcasting_sus.report import generate_report

try:  # 1.2.1+: resolve o ArviZ 1.x internamente
    from nowcasting_sus import save_inference
except ImportError:  # 1.2.0: grava direto pelo backend NetCDF
    save_inference = None

print("python ", sys.version.split()[0])
print("pymc   ", pymc.__version__)
print("arviz  ", arviz.__version__)
print("pandas ", pd.__version__)

In [ ]:
# 2) ENVIAR O ARQUIVO — DBF exportado do SINAN ou CSV (';' ou ',')
up = files.upload()
ARQUIVO = next(iter(up))
print("arquivo:", ARQUIVO)

In [ ]:
# 3) PARÂMETROS
UF = "12"  # 12 = Acre. Prefixo do código IBGE de residência.
CLASSI_FIN = ""  # "" = todas as notificações; "10,11,12" = só confirmados
ANO_MIN = 2026
MAX_DELAY = 30
DRAWS, TUNE, CHAINS = 500, 500, 2  # Colab CPU: ~3–5 min por modelo
SEED = 42

In [ ]:
# 4) LEITURA → CSV mínimo em ISO 8601
# load_sinan NÃO lê DBF: passar o .dbf direto falha com
# "Colunas obrigatórias ausentes ... Colunas disponíveis: ['\x03~\x0b']".
def ler_dbf(caminho):
    """Lê DBF dBase III sem dependência externa."""
    with open(caminho, "rb") as f:
        head = f.read(32)
        nrec = struct.unpack("<I", head[4:8])[0]
        hlen = struct.unpack("<H", head[8:10])[0]
        rlen = struct.unpack("<H", head[10:12])[0]
        nf = (hlen - 33) // 32
        f.seek(32)
        campos = []
        for _ in range(nf):
            d = f.read(32)
            campos.append(
                (d[:11].split(b"\x00")[0].decode("latin1"), chr(d[11]), d[16])
            )
        f.seek(hlen)
        linhas = []
        for _ in range(nrec):
            rec = f.read(rlen)
            if not rec or rec[:1] == b"\x1a":
                break
            if rec[:1] == b"*":  # registro deletado
                continue
            pos, row = 1, {}
            for nome, _tipo, tam in campos:
                row[nome] = rec[pos : pos + tam].decode("latin1").strip()
                pos += tam
            linhas.append(row)
    return pd.DataFrame(linhas)


if ARQUIVO.lower().endswith(".dbf"):
    raw = ler_dbf(ARQUIVO)
    fmt_in = "%Y%m%d"
else:
    # detecta o separador só pelo cabeçalho (sep=None com engine="python" é lento)
    with open(ARQUIVO, encoding="latin1", errors="replace") as fh:
        primeira = fh.readline()
    sep = max([";", ",", "\t", "|"], key=primeira.count)
    raw = pd.read_csv(ARQUIVO, sep=sep, encoding="latin1", dtype=str, low_memory=False)
    amostra = str(raw["DT_SIN_PRI"].dropna().iloc[0])
    fmt_in = "%Y%m%d" if ("-" not in amostra and "/" not in amostra) else "%d/%m/%Y"

print("bruto:", len(raw), "registros |", raw.shape[1], "colunas | data:", fmt_in)

# filtro de UF pela residência (se não houver ID_MN_RESI, usa ID_MUNICIP)
col_uf = "ID_MN_RESI" if "ID_MN_RESI" in raw.columns else "ID_MUNICIP"
ac = raw[raw[col_uf].astype(str).str[:2] == UF].copy()
print(f"UF {UF} ({col_uf}):", len(ac), "registros")

if CLASSI_FIN and "CLASSI_FIN" in ac.columns:
    permitidos = {c.strip() for c in CLASSI_FIN.split(",")}
    ac = ac[ac["CLASSI_FIN"].isin(permitidos)]
    print("após filtro CLASSI_FIN:", len(ac), ac["CLASSI_FIN"].value_counts().to_dict())

iso = pd.DataFrame(index=ac.index)
for col in ("DT_SIN_PRI", "DT_NOTIFIC"):
    iso[col] = pd.to_datetime(ac[col], format=fmt_in, errors="coerce").dt.strftime(
        "%Y-%m-%d"
    )
CSV_ISO = "/content/sinan_iso.csv"
iso.to_csv(CSV_ISO, index=False)
print(
    "datas inválidas — onset:",
    int(iso["DT_SIN_PRI"].isna().sum()),
    "| notificação:",
    int(iso["DT_NOTIFIC"].isna().sum()),
)
print("snapshot (última notificação):", iso["DT_NOTIFIC"].max())

In [ ]:
# 5) CARREGAR NO PACOTE E MONTAR A MATRIZ
df = load_sinan(CSV_ISO, max_delay=MAX_DELAY, min_year=ANO_MIN)
print(
    "válidos:",
    len(df),
    "| delay mediano:",
    df["delay"].median(),
    "| p90:",
    df["delay"].quantile(0.90),
)

nmat, datas, obs_t, obs_d, counts = prepare_matrix(df, max_delay=MAX_DELAY)
T, D = nmat.shape
print("matriz:", T, "dias x", D, "delays |", len(counts), "células observadas")

In [ ]:
# 6) AJUSTE
modelo = NowcastingModel()
idata = modelo.fit(
    obs_t,
    obs_d,
    counts,
    T=T,
    D=D,
    draws=DRAWS,
    tune=TUNE,
    chains=CHAINS,
    target_accept=0.95,
    progressbar=False,
    random_seed=SEED,
)
print("tipo do retorno:", type(idata).__name__)

# ArviZ 1.x removeu az.to_netcdf — salvar pelo objeto exige h5netcdf + h5py
if save_inference is not None:
    print("salvo em:", save_inference(idata, "/content/idata_base.nc"))
else:
    idata.to_netcdf("/content/idata_base.nc", engine="h5netcdf")
    print("salvo em: /content/idata_base.nc")

resumo = modelo.get_summary()
print(
    "R_hat máx:",
    (resumo.get("r_hat") or {}).get("max"),
    "| ESS mín:",
    (resumo.get("n_eff") or {}).get("min"),
    "| divergências:",
    resumo.get("n_divergences"),
)

In [ ]:
# 7) RESULTADOS — últimos 14 dias e figuras
med, lo, hi = modelo.get_nowcast_ci(0.95)
serie = pd.DataFrame(
    {
        "data": pd.to_datetime(datas).date,
        "observado": nmat.sum(axis=1),
        "nowcast": np.round(med, 1),
        "ic95_inf": np.round(lo, 1),
        "ic95_sup": np.round(hi, 1),
        "nao_notificado": np.round(np.clip(med - nmat.sum(axis=1), 0, None), 1),
    }
)
serie.to_csv("/content/serie_nowcast.csv", index=False)
print(serie.tail(14).to_string(index=False))
print(
    "\ntotal observado:",
    int(nmat.sum()),
    "| total nowcast:",
    round(float(np.sum(med)), 1),
)

plot_nowcasting(
    nmat,
    med,
    lo,
    hi,
    datas,
    title="Nowcasting dengue — 2026",
    save_to="/content/nowcast.png",
    dpi=150,
)
generate_report(
    nmat,
    med,
    lo,
    hi,
    datas,
    agravo="Dengue",
    uf="AC",
    cid="A90",
    total_observado=int(nmat.sum()),
    save_to="/content/boletim.html",
)
display(Image("/content/nowcast.png"))

In [ ]:
# 8) BAIXAR OS RESULTADOS
ARQUIVOS = [
    "/content/serie_nowcast.csv",
    "/content/nowcast.png",
    "/content/boletim.html",
    "/content/idata_base.nc",
]
ZIP = "/content/nowcasting_saida.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for caminho in ARQUIVOS:
        z.write(caminho, arcname=os.path.basename(caminho))
print("zip:", ZIP, "|", os.path.getsize(ZIP), "bytes")
files.download(ZIP)


 


 
E
r
r
o
s
 
c
o
n
h
e
c
i
d
o
s
 
n
e
s
t
e
 
a
m
b
i
e
n
t
e






 
|
 
E
r
r
o
 
|
 
C
a
u
s
a
 
|
 
C
o
r
r
e
ç
ã
o
 
|



 
|
-
-
-
|
-
-
-
|
-
-
-
|



 
|
 
`
A
t
t
r
i
b
u
t
e
E
r
r
o
r
:
 
m
o
d
u
l
e
 
'
a
r
v
i
z
'
 
h
a
s
 
n
o
 
a
t
t
r
i
b
u
t
e
 
'
t
o
_
n
e
t
c
d
f
'
`
 
|
 
A
r
v
i
Z
 
1
.
x
 
r
e
m
o
v
e
u
 
a
 
f
u
n
ç
ã
o
 
|
 
`
s
a
v
e
_
i
n
f
e
r
e
n
c
e
(
)
`
 
(
1
.
2
.
1
+
)
 
o
u
 
`
i
d
a
t
a
.
t
o
_
n
e
t
c
d
f
(
e
n
g
i
n
e
=
"
h
5
n
e
t
c
d
f
"
)
`
 
|



 
|
 
`
V
a
l
u
e
E
r
r
o
r
:
 
c
a
n
n
o
t
 
w
r
i
t
e
 
N
e
t
C
D
F
 
f
i
l
e
s
 
w
i
t
h
 
f
o
r
m
a
t
=
'
N
E
T
C
D
F
4
'
 
b
e
c
a
u
s
e
 
n
o
n
e
 
o
f
 
.
.
.
 
(
n
e
t
C
D
F
4
,
 
h
5
n
e
t
c
d
f
)
`
 
|
 
f
a
l
t
a
 
b
a
c
k
e
n
d
 
|
 
`
p
i
p
 
i
n
s
t
a
l
l
 
h
5
n
e
t
c
d
f
 
h
5
p
y
`
 
|



 
|
 
`
I
m
p
o
r
t
E
r
r
o
r
:
 
N
o
 
m
o
d
u
l
e
 
n
a
m
e
d
 
'
h
5
p
y
'
`
 
|
 
`
h
5
n
e
t
c
d
f
`
 
i
n
s
t
a
l
a
d
o
 
s
e
m
 
`
h
5
p
y
`
 
|
 
i
n
s
t
a
l
a
r
 
o
s
 
d
o
i
s
 
|



 
|
 
`
V
a
l
u
e
E
r
r
o
r
:
 
C
o
l
u
n
a
s
 
o
b
r
i
g
a
t
ó
r
i
a
s
 
a
u
s
e
n
t
e
s
 
.
.
.
 
[
'
\
x
0
3
~
\
x
0
b
'
]
`
 
|
 
p
a
s
s
o
u
 
o
 
`
.
d
b
f
`
 
d
i
r
e
t
o
 
p
a
r
a
 
o
 
`
l
o
a
d
_
s
i
n
a
n
`
 
|
 
c
o
n
v
e
r
t
e
r
 
p
a
r
a
 
C
S
V
 
I
S
O
 
(
c
é
l
u
l
a
 
4
)
 
|



 
|
 
`
F
i
l
e
N
o
t
F
o
u
n
d
E
r
r
o
r
:
 
b
a
n
c
o
.
c
s
v
`
 
|
 
e
x
e
m
p
l
o
 
d
o
 
r
e
p
o
 
e
s
p
e
r
a
v
a
 
a
r
q
u
i
v
o
 
l
o
c
a
l
 
|
 
u
s
a
r
 
a
 
c
é
l
u
l
a
 
d
e
 
u
p
l
o
a
d
 
|



 
|
 
s
a
m
p
l
i
n
g
 
l
e
n
t
o
 
o
u
 
t
r
a
v
a
d
o
 
|
 
v
á
r
i
a
s
 
c
a
d
e
i
a
s
 
e
m
 
C
P
U
 
c
o
m
p
a
r
t
i
l
h
a
d
a
 
|
 
`
C
H
A
I
N
S
=
1
`
 
o
u
 
`
D
R
A
W
S
=
3
0
0
`
 
|



 
|
 
a
v
i
s
o
s
 
`
a
z
.
w
a
i
c
`
 
/
 
`
a
z
.
l
o
o
(
s
c
a
l
e
=
.
.
.
)
`
 
|
 
A
P
I
 
m
u
d
o
u
 
n
o
 
A
r
v
i
Z
 
1
.
x
 
|
 
i
g
n
o
r
a
r
:
 
n
ã
o
 
i
n
t
e
r
r
o
m
p
e
 
|